# Human-in-the-Loop

The Human-in-the-Loop (HITL) pattern integrates human judgment into AI workflows. Agents handle routine cases autonomously, while high-risk actions are gated on explicit human approval.

## Implementation with Flyte v2 + the Agent harness

The earlier version modeled escalation as a **return flag** (`SupportResult(escalate=True)`) that an outer loop had to interpret. This refactor makes it a real gate: the sensitive action — `issue_refund` — is a `@tool(requires_approval=True)`. When the agent decides to call it, the harness **pauses the run** via `flyteplugins-hitl` and waits for a human to approve or deny before the tool executes.

#### ADK vs Flyte v2 + Agent harness

| Aspect | ADK | Flyte v2 + `Agent` harness |
|--------|-----|----------------------------|
| **Escalation trigger** | `escalate_to_human` mock tool | `@tool(requires_approval=True)` — real pause-for-approval |
| **Approval transport** | Not shown | `flyteplugins-hitl` surfaces the request in the UI |
| **On denial** | N/A | Agent recovers and tries another approach |
| **Personalization** | `CallbackContext` injection | Customer context passed in the message |
| **Routine tools** | `troubleshoot_issue`, `create_ticket` | `@tool` functions — schema from type hints |
| **Secrets** | `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm flyteplugins-hitl

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
import uuid
from dataclasses import dataclass, field
from datetime import timedelta

import flyte
from flyte.ai.agents import Agent, AgentResult, tool

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="hitl-agent", python_version=(3, 12))
    .with_pip_packages("litellm", "flyteplugins-hitl")
)

hitl_env = flyte.TaskEnvironment(
    name="hitl_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the customer context

`CustomerContext` is a typed task input — the agent receives the customer's tier and history and personalizes accordingly. There is no `SupportResult(escalate=...)` flag anymore: escalation is no longer a return value to interpret, it's an approval gate on the sensitive tool.

In [ ]:
@dataclass
class CustomerContext:
    """Typed customer context passed to the support agent."""
    name: str
    tier: str = "standard"           # "standard" | "premium" | "enterprise"
    recent_purchases: list[str] = field(default_factory=list)
    support_history: list[str] = field(default_factory=list)

### 5. Define the tools (one gated by human approval)

`troubleshoot_issue` and `create_ticket` are ordinary `@tool`s. `issue_refund` is the sensitive, irreversible action — marked `@tool(requires_approval=True)`, so the harness pauses for a human decision before it runs.

In [ ]:
@tool
def troubleshoot_issue(issue: str, device_type: str = "") -> str:
    """Diagnose a technical issue and return step-by-step troubleshooting instructions.

    Args:
        issue: Description of the technical issue.
        device_type: Type of device, e.g. 'laptop' or 'phone'.
    """
    return (
        f"Troubleshooting for '{issue}' ({device_type or 'device'}): "
        "1) restart the device, 2) update firmware, 3) check cables/connections."
    )


@tool
def create_ticket(issue_type: str, details: str) -> str:
    """Create a support ticket and return its ID.

    Args:
        issue_type: Category of the issue.
        details: Full issue description for the ticket.
    """
    return f"TICKET-{uuid.uuid4().hex[:8].upper()} created for {issue_type}."


@tool(requires_approval=True)
def issue_refund(order_id: str, amount_usd: float) -> str:
    """Issue a refund for an order. High-risk and irreversible — requires human approval.

    Args:
        order_id: The order to refund.
        amount_usd: Refund amount in US dollars.
    """
    return f"Refund of ${amount_usd:.2f} issued for order {order_id}."


support_bot = Agent(
    name="support",
    model="claude-haiku-4-5",
    instructions=(
        "You are a technical support specialist for an electronics company. Personalize using "
        "the customer context provided. Use troubleshoot_issue to diagnose problems and "
        "create_ticket to log unresolved ones. If the customer is clearly entitled to a refund "
        "(defective product, or troubleshooting failed for a premium/enterprise customer), call "
        "issue_refund — this pauses for human approval before executing. If approval is denied, "
        "apologize and create a ticket for manual follow-up instead."
    ),
    tools=[troubleshoot_issue, create_ticket, issue_refund],
    max_turns=10,
)

### 6. Define the support agent task

The task is thin: it packs the customer context into the message and runs the agent. When the agent calls `issue_refund`, the harness blocks the run on human approval — the escalation is enforced by the gate, not by an outer `if result.escalate` branch.

In [ ]:
@hitl_env.task(
    retries=1,
    timeout=timedelta(minutes=10),
    cache=flyte.Cache(behavior="disable"),
)
async def support_agent(issue: str, customer: CustomerContext) -> str:
    """Technical support agent with an approval-gated refund action."""
    context = (
        f"Customer: {customer.name} (tier: {customer.tier}). "
        f"Recent purchases: {', '.join(customer.recent_purchases) or 'none'}. "
        f"Past issues: {', '.join(customer.support_history) or 'none'}.\n\n"
        f"Issue: {issue}"
    )
    result: AgentResult = await support_bot.run.aio(context)
    return result.summary or result.error or ""

### 7. Run locally — the approval gate in action

For each scenario the agent troubleshoots and, where warranted, calls `issue_refund`. That call **pauses the run** until a human approves or denies it in the Flyte UI — no outer escalation loop required.

In [ ]:
SCENARIOS = [
    (
        "My laptop screen flickers and sometimes goes black.",
        CustomerContext(
            name="Alice", tier="premium",
            recent_purchases=["ThinkPad X1 Carbon"],
            support_history=[],
        ),
    ),
    (
        "My phone is completely unresponsive after dropping it in water. I can smell burning. "
        "I want a refund for order ORD-9912.",
        CustomerContext(
            name="Bob", tier="standard",
            recent_purchases=["iPhone 15"],
            support_history=["Battery drain", "Screen crack"],
        ),
    ),
    (
        "My Bluetooth headphones won't connect to my laptop.",
        CustomerContext(
            name="Carol", tier="enterprise",
            recent_purchases=["Sony WH-1000XM5"],
            support_history=[],
        ),
    ),
]

for issue, customer in SCENARIOS:
    run = flyte.run(support_agent, issue=issue, customer=customer)
    run.wait()  # pauses here for human approval if the agent calls issue_refund
    print(f"Customer: {customer.name} | Issue: {issue[:60]}")
    print(f"  -> {run.outputs()[0]}")
    print()

### Running remotely

Remote execution is where the gate shines: when the agent calls `issue_refund`, `flyteplugins-hitl` surfaces the pending tool call (with its arguments) in the Flyte UI, and the run stays parked until a human approves or denies it. The decision — and the agent's recovery on denial — are all visible as nested actions under `agent.run`.

In [ ]:
run = flyte.run(
    support_agent,
    issue="My smart thermostat shows error code E5 and won't heat the house. I'd like a refund for order ORD-4471.",
    customer=CustomerContext(name="Dave", tier="standard", recent_purchases=["Nest Thermostat"]),
)
run.wait()
print(run.outputs()[0])